In [ ]:
import sys
import setuptools._distutils.version
sys.modules['distutils.version'] = setuptools._distutils.version

import undetected_chromedriver as uc
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, StaleElementReferenceException
import pandas as pd
import time
import os

# ── All category URLs (your sitemap) ───────────────────────────────────────────
URLS = [
    "https://www.carousell.com.my/categories/video-gaming-697/",
    "https://www.carousell.com.my/categories/video-gaming-697/video-games-consoles-699/",
    "https://www.carousell.com.my/categories/video-gaming-697/video-games-consoles-699/playstation-6003/",
    "https://www.carousell.com.my/categories/video-gaming-697/video-games-consoles-699/nintendo-6004/",
    "https://www.carousell.com.my/categories/video-gaming-697/video-games-consoles-699/xbox-6005/",
    "https://www.carousell.com.my/categories/video-gaming-697/video-games-consoles-699/others-6006/",
    "https://www.carousell.com.my/categories/video-gaming-697/video-games-700/",
    "https://www.carousell.com.my/categories/video-gaming-697/video-games-700/playstation-6007/",
    "https://www.carousell.com.my/categories/video-gaming-697/video-games-700/nintendo-6008/",
    "https://www.carousell.com.my/categories/video-gaming-697/video-games-700/xbox-6009/",
    "https://www.carousell.com.my/categories/video-gaming-697/video-games-700/others-6010/",
    "https://www.carousell.com.my/categories/video-gaming-697/gaming-accessories-702/",
    "https://www.carousell.com.my/categories/video-gaming-697/gaming-accessories-702/controllers-6011/",
    "https://www.carousell.com.my/categories/video-gaming-697/gaming-accessories-702/cases-covers-6012/",
    "https://www.carousell.com.my/categories/video-gaming-697/gaming-accessories-702/cables-chargers-6013/",
    "https://www.carousell.com.my/categories/video-gaming-697/gaming-accessories-702/virtual-reality-6014/",
    "https://www.carousell.com.my/categories/video-gaming-697/gaming-accessories-702/interactive-gaming-figures-6015/",
    "https://www.carousell.com.my/categories/video-gaming-697/gaming-accessories-702/in-game-products-6016/",
    "https://www.carousell.com.my/categories/video-gaming-697/gaming-accessories-702/game-gift-cards-accounts-6017/",
    "https://www.carousell.com.my/categories/audio-595/",
    "https://www.carousell.com.my/categories/audio-595/earphones-6018/",
    "https://www.carousell.com.my/categories/audio-595/headphones-headsets-6019/",
    "https://www.carousell.com.my/categories/audio-595/microphones-6020/",
    "https://www.carousell.com.my/categories/audio-595/voice-recorders-6021/",
    "https://www.carousell.com.my/categories/audio-595/portable-music-players-6022/",
    "https://www.carousell.com.my/categories/audio-595/portable-audio-accessories-6023/",
    "https://www.carousell.com.my/categories/audio-595/soundbars-speakers-amplifiers-6024/",
    "https://www.carousell.com.my/categories/audio-595/other-audio-equipment-6025/",
    "https://www.carousell.com.my/categories/photography-6/",
    "https://www.carousell.com.my/categories/photography-6/cameras-6026/",
    "https://www.carousell.com.my/categories/photography-6/lens-kits-6027/",
    "https://www.carousell.com.my/categories/photography-6/drones-6028/",
    "https://www.carousell.com.my/categories/photography-6/video-cameras-6029/",
    "https://www.carousell.com.my/categories/photography-6/photography-accessories-6030/",
    "https://www.carousell.com.my/categories/photography-6/photography-accessories-6030/camera-bags-carriers-6031/",
    "https://www.carousell.com.my/categories/photography-6/photography-accessories-6030/gimbals-stabilisers-6032/",
    "https://www.carousell.com.my/categories/photography-6/photography-accessories-6030/lighting-studio-equipment-6033/",
    "https://www.carousell.com.my/categories/photography-6/photography-accessories-6030/flashes-6034/",
    "https://www.carousell.com.my/categories/photography-6/photography-accessories-6030/tripods-monopods-6035/",
    "https://www.carousell.com.my/categories/photography-6/photography-accessories-6030/dry-boxes-cabinets-6036/",
    "https://www.carousell.com.my/categories/photography-6/photography-accessories-6030/batteries-chargers-6037/",
    "https://www.carousell.com.my/categories/photography-6/photography-accessories-6030/other-photography-accessories-6038/",
]

# ── Config ─────────────────────────────────────────────────────────────────────
CSV_FILE        = "carousell_data_pravin.csv"
PROGRESS_FILE   = "carousell_progress_pravin.txt"
MAX_RECORDS     = 100_000
SAVE_EVERY      = 50
SCROLL_PAUSE    = 2.5
MAX_STALE       = 10

# ── Browser setup ──────────────────────────────────────────────────────────────
options = uc.ChromeOptions()
# options.add_argument("--headless=new")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument(
    "--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
    "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36"
)

driver = uc.Chrome(options=options, version_main=147)
print("✅ Browser started!")

# ── Resume: load existing CSV & completed-URL log ──────────────────────────────
results  = []
seen_ids = set()

if os.path.exists(CSV_FILE):
    print(f"📂 Found existing file – resuming from '{CSV_FILE}'…")
    df_old   = pd.read_csv(CSV_FILE)
    results  = df_old.to_dict(orient="records")
    if "listing_id" in df_old.columns:
        seen_ids = set(df_old["listing_id"].dropna().astype(str))
        print(f"   ↳ Resumed with {len(seen_ids)} previously scraped cards.")
    else:
        print("   ⚠️  Old CSV missing 'listing_id' – duplicates may occur.")
else:
    print("🚀 Starting fresh scrape.")

completed_urls = set()
if os.path.exists(PROGRESS_FILE):
    with open(PROGRESS_FILE, "r") as f:
        completed_urls = set(line.strip() for line in f if line.strip())
    print(f"   ↳ {len(completed_urls)} URLs already completed – will skip them.\n")


def save():
    pd.DataFrame(results).to_csv(CSV_FILE, index=False)
    print(f"   💾 Saved {len(results):,} records → {CSV_FILE}")


def mark_url_done(url):
    with open(PROGRESS_FILE, "a") as f:
        f.write(url + "\n")


def get_category_from_page():
    """
    Extract the full category breadcrumb from the page.
    On Carousell category pages the breadcrumb looks like:
    Home > Computers & Tech > Parts & Accessories > Chargers
    Falls back to a cleaned URL path if the breadcrumb is missing.
    """
    try:
        # The breadcrumb <ul> typically has class "D_aI_"
        breadcrumb_ul = driver.find_element(By.CSS_SELECTOR, "ul.D_aI_")
        lis = breadcrumb_ul.find_elements(By.TAG_NAME, "li")
        # Skip the first "Home" item, join the rest with " > "
        texts = [li.text.strip() for li in lis[1:] if li.text.strip()]
        if texts:
            return " > ".join(texts)
    except Exception:
        pass

    # Fallback: derive from URL
    try:
        path = driver.current_url.split("/categories/")[1].split("?")[0]
        # Replace hyphens + IDs with spaces, then clean up
        import re
        # Remove the trailing category ID numbers and hyphens
        parts = re.sub(r'-\d+', '', path).strip('-').split('/')
        category = " > ".join(p.strip().replace("-", " ") for p in parts if p.strip())
        return category if category else "Unknown"
    except:
        return "Unknown"


def parse_cards(category_text):
    from bs4 import BeautifulSoup
    soup = BeautifulSoup(driver.page_source, "html.parser")

    cards = soup.select(
        '[data-testid^="listing-card-"]'
        ':not([data-testid="listing-card-text-seller-name"])'
        ':not([data-testid="listing-card-btn-like"])'
    )

    new_records = []
    for card in cards:
        raw_testid = card.get("data-testid", "")
        listing_id = raw_testid.replace("listing-card-", "")

        if not listing_id or listing_id in seen_ids:
            continue

        img_el  = card.select_one('a[href^="/p/"] img')
        product = img_el["alt"].strip() if img_el and img_el.get("alt") else None

        if not product:
            continue

        link_el = card.select_one('a[href^="/p/"]')
        listing_url = (
            "https://www.carousell.com.my" + link_el["href"].split("?")[0]
            if link_el else None
        )

        # ── FIXED: find the price <p> using its title attribute ──
        price_el = card.select_one('p[title^="RM"]')  # adjust if other currencies appear
        price    = price_el["title"].strip() if price_el else None

        # Original (strikethrough) price
        orig_el = card.select_one('span[aria-label^="Stricken Price"] s')
        original_price = orig_el.get_text(strip=True) if orig_el else None

        # Condition: the <p> right after the price container
        condition = None
        if price_el:
            # price_el is usually inside a <div>, e.g. <div class="D_ban ...">
            price_container = price_el.parent
            # Next sibling that is a <p>
            condition_el = price_container.find_next_sibling("p")
            if condition_el:
                condition = condition_el.text.strip()

        seller_el = card.select_one('[data-testid="listing-card-text-seller-name"]')
        seller    = seller_el.text.strip() if seller_el else None

        time_el = card.select_one(".D_aZi p")   # may need adjusting; fallback described below
        time_posted = time_el.text.strip() if time_el else None

        like_el = card.select_one('[data-testid="listing-card-btn-like"] span')
        likes   = like_el.text.strip() if like_el else "0"

        # Buyer Protection badge
        bp_badge = card.select_one('div.D_uT p')
        buyer_protection = True if (bp_badge and 'Buyer Protection' in bp_badge.get_text()) else False

        # Delivery info (e.g., "Free delivery")
        delivery_el = card.select_one('div.D_afi p')
        delivery_info = delivery_el.get_text(strip=True) if delivery_el else None

        seen_ids.add(listing_id)
        new_records.append({
            "listing_id":      listing_id,
            "product":         product,
            "price":           price,
            "original_price":  original_price,
            "condition":       condition,
            "seller":          seller,
            "time_posted":     time_posted,
            "likes":           likes,
            "listing_url":     listing_url,
            "source_url":      driver.current_url,
            "category":        category_text,
            "buyer_protection": buyer_protection,
            "delivery_info":   delivery_info,
        })

    return new_records


def scrape_url(url):
    """Load one category URL, get its breadcrumb, then scroll to extract listings."""
    global results

    print(f"\n{'='*70}")
    print(f"🌐 Scraping: {url}")
    print(f"   Progress: {len(results):,} total records so far")
    print(f"{'='*70}")

    driver.get(url)
    try:
        WebDriverWait(driver, 20).until(
            EC.presence_of_element_located(
                (By.CSS_SELECTOR, '[data-testid^="listing-card-"]')
            )
        )
        print("   ✅ Page loaded.")
    except TimeoutException:
        print("   ❌ Timed out waiting for cards – skipping this URL.")
        return

    time.sleep(2)  # let lazy content settle

    # 👇 Extract category once for the whole page
    category_text = get_category_from_page()
    print(f"   📁 Category: {category_text}")

    stale_count      = 0
    last_save_count  = len(results)

    while len(results) < MAX_RECORDS:

        new_records = parse_cards(category_text)   # 👈 pass category in

        if new_records:
            results.extend(new_records)
            stale_count = 0
            print(f"   📊 Total: {len(results):,}  (+{len(new_records)} new)")

            if len(results) - last_save_count >= SAVE_EVERY:
                save()
                last_save_count = len(results)
        else:
            stale_count += 1
            print(f"   ⏳ No new cards ({stale_count}/{MAX_STALE}) – scrolling…")

        if stale_count >= MAX_STALE:
            print(f"   ✅ Stale limit hit – done with this URL.")
            break

        # Click "Show more results" if available
        try:
            show_more = driver.find_element(
                By.XPATH, "//button[contains(., 'Show more results')]"
            )
            if show_more.is_displayed():
                driver.execute_script("arguments[0].scrollIntoView(true);", show_more)
                time.sleep(0.5)
                show_more.click()
                print("   👉 Clicked 'Show more results'")
                time.sleep(SCROLL_PAUSE)
        except Exception:
            pass

        # Scroll to bottom
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(SCROLL_PAUSE)
        driver.execute_script("window.scrollBy(0, -300);")
        time.sleep(0.3)
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(0.5)


# ── Main loop ──────────────────────────────────────────────────────────────────
urls_to_scrape = [u for u in URLS if u not in completed_urls]
print(f"\n📋 {len(urls_to_scrape)} URLs to scrape "
      f"({len(completed_urls)} already done).\n")

for i, url in enumerate(urls_to_scrape, 1):
    print(f"\n[{i}/{len(urls_to_scrape)}]", end="")
    scrape_url(url)
    mark_url_done(url)
    save()

    if len(results) >= MAX_RECORDS:
        print(f"\n🏁 Hit MAX_RECORDS ({MAX_RECORDS:,}) – stopping early.")
        break

    time.sleep(3)

# ── Final save ────────────────────────────────────────────────────────────────
driver.quit()
save()
print(f"\n🏁 Finished. {len(results):,} records saved to '{CSV_FILE}'.")